Every machine learning textbook teaches the same U-curve: as capacity grows, held-out error falls to a minimum, then climbs as the model starts fitting noise. The bias-variance tradeoff explains that shape, and a validation set finds the turn.

Deep networks break the picture. With orders of magnitude more parameters than data points, they fit their training samples exactly (noise included) and still predict new data well. Their held-out error can even fall a second time after rising near perfect fitting. The pattern is called *double descent* [1]. The trouble concentrates near the interpolation threshold, where capacity first suffices to fit every training point. A small change in the sample can move predictions sharply, and held-out error can spike. Just past the threshold, many exact fits exist. The training procedure has to choose among them.

Parameter count, the classical proxy for capacity, is a poor guide to what these systems actually do.

Language models sit on the same axis with complications of their own. They are neural sequence predictors fitted on raw text [2], steered toward new tasks by instructions alone or by examples placed in the input [3, 4]. Zero-shot and *in-context learning* made the prompt itself the interface. Assistants add supervised fine-tuning [5] and training against human preferences or verifiable rewards [6]. The statistical surprises reappear, sharper. Double descent has been mapped as a joint property of capacity and training time [7, 8]. Small algorithmic tasks supplied the cleanest departure. A network fits first and generalizes only after many further steps, a delay known as *grokking* [9].

At scale, the prompt-facing measurements turn volatile. Chain-of-thought prompting exposes intermediate steps that a single answer hides [10], and some benchmark scores sit near chance before rising steeply, the pattern called *emergent abilities* [11]. A discontinuous metric can make a gradual underlying change look sudden [12]. The smooth measurements live elsewhere. Average loss on held-out text declines predictably across orders of magnitude of compute. The same model looks volatile through one instrument and steady through the other. Compute spent at test time, on repeated attempts or search, is a budget of its own. Post-training against verifiable rewards [6] can teach a system to spend it. Evaluation changes too. Reasoning traces, checkable outcomes, and multi-step tool use need more than a single label comparison. A response can follow every instruction in the prompt and still misstate the situation it concerns, so correctness becomes, in part, a deployment decision.

What follows goes from the anomaly to the newest compute budget, through deep networks and the Transformer [13].

## 1. AlexNet and the limit of the U-curve

A Welch Labs video on double descent [14] sent me back to AlexNet's 2012 ImageNet result. The network had about 60 million parameters and learned from roughly 1.2 million labeled images [15]. By the classical account, that ratio should have set off alarms. A model with more parameters than examples has enough freedom to memorize its training set. AlexNet won the challenge anyway. One model cannot trace a curve, so the result proves nothing by itself. But it gives the familiar expectation a counterexample to explain. A model with more parameters than examples generalized anyway.

Generalization error, the error a model makes on unseen samples rather than the ones it trained on, is governed by the bias-variance tradeoff. Bias is the error of a model too simple to represent the pattern. Variance is what comes from the particular training sample, noise included. Put them together and the U appears. Added parameters lower bias until rising variance wins the trade. AlexNet looked like a violation of exactly that. The first explanation on offer was explicit regularization. Dropout, weight decay, and data augmentation were holding variance down, the reasoning went, because 60 million parameters would otherwise memorize everything [15].

In 2017, a single experiment broke the story. Replace the image labels with random noise and train standard convolutional networks on the result. They still reach zero training error [16]. A model that can fit random labels has enough capacity to memorize its training set, with or without explicit regularizers. The other half of the experiment put the true labels back, and the same networks generalized to held-out images even when every standard regularizer was removed [16]. Regularization remained a useful tuning knob.

Then came the variance measurement. Across fully connected networks of different widths, prediction variance falls as they widen, the opposite of what the classical picture predicts [17].

![Prediction variance across network widths.](assets/figures/variance-by-source.png)

*Schematic of Neal et al.'s width experiments [17]: total prediction variance falls as networks widen. The vertical line marks the interpolation threshold; the curves show trends, not measured values.*

In settings where a model fits its training set fully, held-out error falls again [1]. Change the variable from size to training time and related patterns appear [8]. The classical U-curve is the left-hand side of that longer trajectory. At the interpolation threshold, capacity is just enough to force training error to zero. The fit there is pinned to every training point, with little redundancy left to average away noise, so small changes in the sample move predictions sharply. Just past the threshold, a whole family of zero-error fits exists. The optimizer is not neutral among those fits. It favors the smoother, lower-norm members. That preference can produce better held-out error. The full trajectory is called double descent: a first descent, a peak at interpolation, a second descent [1, 8].

The two panels place the classical curve beside the same trajectory pushed past interpolation.

![The classical curve beside a possible double-descent trajectory.](assets/figures/double-descent.png)

*The classical picture (left) and a possible double-descent trajectory (right). Left: expected error against model flexibility, with bias and variance. Right: training error (dashed) reaches zero at the interpolation threshold while held-out error (solid) can spike and then fall again. Both panels are schematics, not AlexNet measurements. After Belkin et al. [1] and Nakkiran et al. [8].*

The peak is not universal. Cleaner labels often flatten it into a plateau near interpolation [8], and the shape depends on the data, the model family, and the training budget. A single model cannot establish it. To look for double descent, train a range of sizes on the same data and plot training and held-out error together, marking where training error first reaches zero. Then ask whether held-out error rises and falls again. If training time is the variable, the comparison is different. The axis should say so.

> **Original figures:** The images above are schematics, not reproductions. For measured results, see Zhang et al. ([PDF](https://arxiv.org/pdf/1611.03530)), Neal et al. ([PDF](https://arxiv.org/pdf/1810.08591)), Belkin et al. ([paper](https://www.pnas.org/doi/10.1073/pnas.1903070116)), and Nakkiran et al. ([PDF](https://arxiv.org/pdf/1912.02292)).

-----

## 2. Why zero-error fits generalize

The interpolation threshold is easiest to see with a polynomial. Suppose we observe eight distinct input values $x_i$ and their targets $y_i$. A polynomial of degree at most seven threads all eight points. Allow one more degree and nothing pins the fit down. There are more coefficients than observations. If $p_7(x)$ fits the eight points, so does

$$p_c(x)=p_7(x)+c\prod_{i=1}^{8}(x-x_i),$$

for any value of $c$. The added term vanishes at every observed input, so each member of the family is an exact fit. Between the points, the curves are free to disagree. An exact training fit does not identify a prediction at a new input.

All three of the fits below pass through the same eight points. Between the points, they disagree.

![Three exact fits through the same eight points.](assets/figures/polynomial-fits.png)

*The eight observed points admit more than one exact fit. The dashed curve is the unique degree-7 interpolant; the two solid curves add the term $c\prod_{i=1}^{8}(x-x_i)$ for different values of $c$. All three pass through every point and disagree between them. A constructed example, not measured data.*

Every value of the constant gives another exact fit. The curve highlighted below is the smoothest of the family.

![The full family of exact fits, with the smoothest member highlighted.](assets/figures/polynomial-family.png)

*The full family of exact fits for the same eight points: one curve per value of $c$, all passing through every observed point. The highlighted member is the smoothest one—the kind of fit the training procedure's preference tends toward; its neighbors trade smoothness for brittle wiggles. A constructed example, not measured data.*

Every member of this family fits the eight points exactly, so the training data cannot choose among them. The training procedure does the choosing, and it is not neutral. It tends toward the smooth, low-norm members and away from the brittle, wiggly ones. Change the starting point, the regularization, or the parameterization, and a different member wins.

A deep network offers no closed-form $p_c$ to consult. The same choice happens anyway, inside training dynamics that are messier than any polynomial family.

Even with every explicit penalty switched off, gradient descent does not wander blindly through the zero-error fits. It starts near zero and drifts toward solutions of the smallest magnitude, an effect known as *implicit regularization*. For linear classifiers on separable data, the iterates provably converge in direction to the maximum-margin separator [18].

Stretch the same dynamic over training time and it shows up as grokking. Train small networks on algorithmic tasks and watch the curves separate. Training accuracy approaches perfection while validation sits at chance—then, after many further steps, validation suddenly improves [9]. The late shift traces to compression. A bulky, high-norm memorizing circuit gives way to a smaller, generalizing one [19]. The same model moves from one regime to the other, a transition known as epoch-wise double descent.

The spike comes from the geometry of the fit. At the interpolation threshold, a linear fit has to invert a nearly singular matrix. The smallest singular values collapse toward zero. They correspond to the directions in which the data vary least, and fitting divides by them. That division amplifies the tiniest label noise into a large swing [20]. Move well past the threshold and the geometry turns favorable. The directions that overparameterization adds lift those singular values off zero, and predictions stabilize. The argument is exact for linear least squares and an analogy for deep networks. It also explains why the spike lands exactly where training error first reaches zero.

Some noise does get memorized. The theory of *benign overfitting* explains why that is usually harmless. A model's capacity splits in two. A few dominant directions carry the signal, and a long tail of low-variance directions absorbs the noise [21]. Since new inputs barely reach that tail, the memorized noise has little effect on the prediction.

The grokking phase shift, noisy memorization giving way to consolidated, low-norm circuits, underlies modern Parameter-Efficient Fine-Tuning (PEFT). Pre-training compresses a language model's core knowledge onto a low-dimensional manifold [22]. One consequence follows directly. Adapting the model to a new domain, or updating its facts, modifies a low-rank slice of its weights rather than retraining all of them. Low-rank adapters (LoRA) injected into the model's projections adjust the consolidated knowledge [23]. Once compression has done its work, a fraction of the model's degrees of freedom suffices for adaptation, an economy that holds at billion-parameter scale.

## 3. Smooth loss, bumpy benchmarks

A model's average next-token loss can improve while its answers to one particular request get worse. The two are different measurements, and the second needs its own notation.

Let $X$ be the context and $T$ the task stated in the prompt, and let $Y$ be the answer being evaluated. When $Y$ is a numeric score, a task outcome can be studied through $\mathbb{E}[Y \mid X,T]$. Generated text is handled as probabilities over possible responses instead.

A single language model can be directed toward several tasks through text alone, with no weight update [3]. The task changes with the prompt even when the model does not.

A capability is emergent when its measured performance sits near chance in smaller models and rises substantially in larger ones [11]. The examples are specific prompted benchmarks. No single threshold governs the acquisition of every new skill.

Knowing that a model has lower average prediction loss still does not tell you how well it will handle a particular request.

![Benchmark scores rising against training compute.](assets/figures/emergent-abilities-across-scale.png)

*Eight prompted-task benchmarks across model families; several scores rise after a low-scoring range. Figure from [Schaeffer, Miranda and Koyejo](https://arxiv.org/abs/2304.15004), reproducing results from Wei et al. [11]; also [shown on ResearchGate](https://www.researchgate.net/figure/Emergent-abilities-of-large-language-models-Language-model-families-display-sharp-and_fig1_370417780). The horizontal axis is training FLOPs; the scores describe behavior, leaving the mechanism open.*

Some requests show a less orderly curve. On eleven datasets, task performance *decreases* as model scale increases, a pattern called *inverse scaling* [24]. A possible source is a misleading but easier task in the prompt. Another is demonstrations that encourage the wrong continuation. The finding is local to the task and its evaluation. Particular prompts can expose failures that aggregate loss conceals. As models grow larger, six of those eleven tasks recover [25]. Their performance first falls and then climbs. Four keep degrading with scale, and one improves throughout.

Expressed as error, the recovering tasks rise and then fall across the sizes examined. The shape resembles the recovery phase of double descent. No interpolation threshold or shared mechanism has been established for these tasks. Scale, training, and the prompt change together in these evaluations. Those sources explain only part of the reversal.

Even a sharp-looking jump deserves care. A discontinuous score such as exact match can turn smooth growth in the underlying answer probabilities into an abrupt benchmark curve [12]. For a multi-step answer, the probability of each correct token, $P(y_t\mid y_{<t},X,T)$, can rise without the whole response crossing the exact-match boundary. The jump can still reflect real new behavior. The plot alone cannot say whether the underlying change was sudden or gradual.

Average loss on held-out text declines smoothly while the task scores stay volatile. The training regime explains part of the difference. A classical benchmark cycles over a finite dataset until the model fits it. That exact fit produces the interpolation peak of Section 1.

Frontier language models train in a streaming regime over trillions of tokens and rarely read the same sequence twice. With no fixed sample to fit again and again, the interpolation threshold never forms. Aggregate loss follows a smooth trend as size and token count change. Repetition does occur, and language models can overfit. Within the measured range, the loss stays predictable.

That trend has a fitted form. *Training Compute-Optimal Large Language Models* gives the relation, commonly called the Chinchilla scaling law [26], which predicts the expected loss of a planned run before training begins:

$$L(N,D)=E+\frac{A}{N^{\alpha}}+\frac{B}{D^{\beta}}.$$

Here $N$ counts non-embedding parameters, $D$ counts training tokens, and $L$ is the expected cross-entropy loss. The fitted floor $E$ summarizes what remains in this model of the data, and it should not be read as a universal constant for natural language. Parameter and data terms decline as power laws, with exponents of about $\alpha=0.34$ and $\beta=0.28$. The relation is empirical and holds within the range that was studied.

Under the approximate training-compute relation $C\approx6ND$, the fit favors growing tokens and parameters together when *training loss under a fixed training budget* is the objective. The Chinchilla experiments put the useful balance near 20 training tokens per parameter in their setting [26]. That ratio is about training compute alone, and it leaves out the cost of serving the model repeatedly afterward.

![Selected models by parameter count and training-token-to-parameter ratio.](assets/figures/chinchilla-data-optimal-scaling.png)

*Token-to-parameter ratios for selected models, drawn from the estimates compiled by [Alan D. Thompson, LifeArchitect.ai](https://lifearchitect.ai/chinchilla/) (December 2023). This is an annotated comparison, not Hoffmann et al.'s fitted loss surface: some model values are estimates, and moving to the right does not by itself imply a poor deployment choice.*

So a compact model can be trained on far more data than the training-compute optimum suggests. Llama 3 8B was trained on more than 15 trillion tokens [27], roughly 90 times the 20-token-per-parameter reference.

The trade is deliberate. Extra compute before deployment yields a smaller model to serve afterward.

Over-training can improve a compact model's performance, though whether it crossed a particular emergent threshold must still be checked on the prompted task.

A second trade happens after training. A single generation produces one sample from the model's next-token distribution, and pass@1 scores that one attempt [28].

Inference-time compute buys more attempts. Generate N candidate answers, then select among them.

Where the task admits an automatic check, unit tests or an exact answer, a trained verifier can rank the candidates. Returning the highest-scored sample is best-of-N selection [29].

For reasoning tasks, no learned verifier is needed at all. Sample several chains of thought and return the majority answer, an approach called self-consistency [30]. The weights stay fixed in every scheme. What varies is how many samples the system buys and how it chooses between them. Selection can work below the level of the full answer. A step-level verifier, a process reward model scoring each intermediate step, can prune failing branches and continue along the rest, turning one pass into a search over reasoning paths. The trade is explicit. Extra tokens and latency buy more attempts or deeper search, and the returns concentrate on verifiable tasks. Without a reliable check, more samples reproduce the same errors at higher cost.

The effect on error is structural. At fixed weights, a larger inference budget changes the distribution over answers and, on verifiable tasks, reduces the variance across attempts. Under a latency and cost constraint, correctness becomes partly purchasable.

In the studied setting, allocating the budget well let a smaller model surpass one fourteen times larger at matched compute, on problems the base model already solves some of the time [31]. The behavior being scaled can itself be trained.

DeepSeek-R1 was optimized against verifiable rewards, and the resulting policy produces longer, self-checking reasoning traces [6]. The pre-training law and the inference budget measure different quantities. The law forecasts the loss of the model being trained. Accuracy on a prompted task depends on how that model is queried and checked.

## 4. Where to spend compute

The old posture treated capacity as a hazard. Add just enough parameters to capture the signal and regularize hard. Stop when validation error first turns upward. AlexNet through the Chinchilla fit showed where that posture breaks down. Overparameterization supplies the headroom, and compression does the rest. Gradient descent turns noisy memorization into smooth, low-norm solutions. Low-rank adapters exploit those solutions, and a prompt can specify a new task without touching the weights at all, the conditional expectation $\mathbb{E}[Y \mid X,T]$ of Section 3.

For practitioners, held-out error is no longer a static number fixed in the weights when training ends. It is now dynamic at three levels. It moves with the training regime, where single-pass streaming and inference-optimal over-training follow predictable power laws. The task moves it too. Performance can fall and later recover across scale, and a discontinuous metric can make a gradual gain look sudden. At runtime it moves again, where search, verification, and backtracking make test-time compute an active dial for reducing error, when the task and the checking support it.

Track continuous token loss beside exact-match scores during and after training. The first declines while the second stalls and jumps, so reading only the second turns steady progress into apparent failure. A long validation plateau can precede a late drop. Keep measuring through it. For prediction under a defined sampling process, cross-validation, RMSE, and AUC remain the right instruments. Once the unit of evaluation is a trajectory, with a model calling tools and changing state across steps, correctness shows up in outcomes and constraint checks.

The work now includes deciding where capability lives. It can sit compressed in the weights from pre-training, added through low-rank updates afterward, or searched and verified at the moment of inference. That choice is a compute budget question. The old safeguards keep their place. Evaluation now covers trajectories as well as predictions.

## References

1. Belkin, M., Hsu, D., Ma, S. and Mandal, S. (2019). "Reconciling modern machine-learning practice and the bias–variance trade-off." https://www.pnas.org/doi/10.1073/pnas.1903070116
2. Bengio, Y., Ducharme, R., Vincent, P. and Jauvin, C. (2003). "A Neural Probabilistic Language Model." https://www.jmlr.org/papers/v3/bengio03a.html
3. Radford, A. et al. (2019). "Language Models are Unsupervised Multitask Learners." https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
4. Brown, T. B. et al. (2020). "Language Models are Few-Shot Learners." https://arxiv.org/abs/2005.14165
5. Ouyang, L. et al. (2022). "Training language models to follow instructions with human feedback." https://arxiv.org/abs/2203.02155
6. DeepSeek-AI (2025). "DeepSeek-R1 incentivizes reasoning in LLMs through reinforcement learning." https://www.nature.com/articles/s41586-025-09422-z
7. Lafon, M. and Thomas, A. (2024). "Understanding the Double Descent Phenomenon in Deep Learning." https://arxiv.org/abs/2403.10459
8. Nakkiran, P. et al. (2020). "Deep Double Descent: Where Bigger Models and More Data Hurt." https://arxiv.org/abs/1912.02292
9. Power, A. et al. (2022). "Grokking: Generalization Beyond Overfitting on Small Algorithmic Datasets." https://arxiv.org/abs/2201.02177
10. Wei, J. et al. (2022). "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models." https://arxiv.org/abs/2201.11903
11. Wei, J. et al. (2022). "Emergent Abilities of Large Language Models." https://arxiv.org/abs/2206.07682
12. Schaeffer, R., Miranda, B. and Koyejo, S. (2023). "Are Emergent Abilities of Large Language Models a Mirage?" https://arxiv.org/abs/2304.15004
13. Vaswani, A. et al. (2017). "Attention Is All You Need." https://arxiv.org/abs/1706.03762
14. Welch Labs (2025). "What the Books Get Wrong about AI [Double Descent]." YouTube. https://www.youtube.com/watch?v=z64a7USuGX0
15. Krizhevsky, A., Sutskever, I. and Hinton, G. E. (2012). "ImageNet Classification with Deep Convolutional Neural Networks." https://papers.nips.cc/paper_files/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html
16. Zhang, C., Bengio, S., Hardt, M., Recht, B. and Vinyals, O. (2017). "Understanding deep learning requires rethinking generalization." https://arxiv.org/abs/1611.03530
17. Neal, B., Mittal, S., Baratin, A., Tantia, V., Scicluna, M., Lacoste-Julien, S. and Mitliagkas, I. (2019). "A Modern Take on the Bias-Variance Tradeoff in Neural Networks." https://arxiv.org/abs/1810.08591
18. Soudry, D., Hoffer, E., Nacson, M. S., Gunasekar, S. and Srebro, N. (2018). "The Implicit Bias of Gradient Descent on Separable Data." Journal of Machine Learning Research 19(70), 1–57. https://arxiv.org/abs/1710.10345
19. Liu, Z., Kitouni, O., Nolte, N., Michaud, E. J., Tegmark, M. and Williams, M. (2022). "Towards Understanding Grokking: An Effective Theory of Representation Learning." Advances in Neural Information Processing Systems 35, 34651–34663. https://arxiv.org/abs/2205.10343
20. Hastie, T., Montanari, A., Rosset, S. and Tibshirani, R. J. (2022). "Surprises in High-Dimensional Ridgeless Least Squares Interpolation." The Annals of Statistics 50(2), 949–986. https://arxiv.org/abs/1903.08560
21. Bartlett, P. L., Long, P. M., Lugosi, G. and Tsigler, A. (2020). "Benign Overfitting in Linear Regression." Proceedings of the National Academy of Sciences 117(48), 30063–30070. https://arxiv.org/abs/1906.11300
22. Aghajanyan, A., Zettlemoyer, L. and Gupta, S. (2021). "Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning." Proceedings of the 59th Annual Meeting of the Association for Computational Linguistics (ACL), 7319–7328. https://arxiv.org/abs/2012.13255
23. Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L. and Chen, W. (2022). "LoRA: Low-Rank Adaptation of Large Language Models." International Conference on Learning Representations (ICLR). https://arxiv.org/abs/2106.09685
24. McKenzie, I. R. et al. (2023). "Inverse Scaling: When Bigger Isn't Better." Transactions on Machine Learning Research. https://arxiv.org/abs/2306.09479
25. Wei, J., Kim, N., Tay, Y. and Le, Q. V. (2023). "Inverse Scaling Can Become U-Shaped." Transactions on Machine Learning Research. https://arxiv.org/abs/2211.02011
26. Hoffmann, J. et al. (2022). "Training Compute-Optimal Large Language Models." Advances in Neural Information Processing Systems 35, 30016–30030. https://arxiv.org/abs/2203.15556
27. Dubey, A. et al. (2024). "The Llama 3 Herd of Models." arXiv. https://arxiv.org/abs/2407.21783
28. Chen, M. et al. (2021). "Evaluating Large Language Models Trained on Code." arXiv. https://arxiv.org/abs/2107.03374
29. Cobbe, K. et al. (2021). "Training Verifiers to Solve Math Word Problems." arXiv. https://arxiv.org/abs/2110.14168
30. Wang, X. et al. (2023). "Self-Consistency Improves Chain of Thought Reasoning in Language Models." International Conference on Learning Representations (ICLR). https://arxiv.org/abs/2203.11171
31. Snell, C., Lee, J., Xu, K. and Kumar, A. (2024). "Scaling LLM Test-Time Compute Optimally can be More Effective than Scaling Model Parameters." arXiv. https://arxiv.org/abs/2408.03314